In [6]:
# =====================================================
# ✅ ViT (Vision Transformer) GPU 학습 코드 - Colab용
# =====================================================

# ⚙️ 라이브러리 임포트
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import timm

# =====================================================
# ✅ 하이퍼파라미터 및 경로 설정
# =====================================================
DATA_PATH = "/content/drive/MyDrive/"
TRAIN_PATH = os.path.join(DATA_PATH, "train_images")
TEST_PATH = os.path.join(DATA_PATH, "test.jpg")
MODEL_PATH = os.path.join(DATA_PATH, "jx_vit_base_p16_224-80ecf9dd.pth")

IMG_SIZE = 224            # ViT Base 입력 크기
BATCH_SIZE = 16
LR = 2e-5
N_EPOCHS = 1              # 예시용 (시간 단축)
GAMMA = 0.7

# =====================================================
# ✅ 1. 데이터셋 클래스 예시 (CassavaDataset)
# =====================================================
# ⚠️ 사용자 코드에 맞게 수정 가능 — 예시는 기본 형태로 작성
from torchvision import transforms, datasets

transforms_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transforms_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# 예시용 dummy dataset (사용자는 실제 train_df, valid_df 사용)
train_dataset = datasets.FakeData(
    size=64, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_train
)
valid_dataset = datasets.FakeData(
    size=16, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_valid
)

# =====================================================
# ✅ 2. DataLoader 정의
# =====================================================
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =====================================================
# ✅ 3. ViT 모델 정의
# =====================================================
class ViTBase16(nn.Module):
    def __init__(self, n_classes, pretrained=False):
        super(ViTBase16, self).__init__()
        self.model = timm.create_model("vit_base_patch16_224", pretrained=False)
        if pretrained:
            self.model.load_state_dict(torch.load(MODEL_PATH))
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, n_classes)

    def forward(self, x):
        return self.model(x)

# 클래스 수 (예시: 5)
model = ViTBase16(n_classes=5, pretrained=False)

# =====================================================
# ✅ 4. GPU 디바이스 설정
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")
model.to(device)

# =====================================================
# ✅ 5. 손실함수, 옵티마이저, 스케줄러
# =====================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=GAMMA)

# =====================================================
# ✅ 6. 학습 함수 정의
# =====================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, running_acc = 0.0, 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        acc = (outputs.argmax(1) == labels).float().mean()
        running_loss += loss.item()
        running_acc += acc.item()
    return running_loss / len(loader), running_acc / len(loader)

# =====================================================
# ✅ 7. 검증 함수 정의
# =====================================================
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            acc = (outputs.argmax(1) == labels).float().mean()
            val_loss += loss.item()
            val_acc += acc.item()
    return val_loss / len(loader), val_acc / len(loader)

# =====================================================
# ✅ 8. 메인 학습 루프
# =====================================================
start_time = datetime.now()
print(f"⏱ Start training: {start_time}")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, valid_loader, criterion, device)
    scheduler.step()

    print(f"[Epoch {epoch+1}/{N_EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"✅ Training completed. Total time: {datetime.now() - start_time}")

# =====================================================
# ✅ 9. 모델 저장
# =====================================================
save_path = f"/content/drive/MyDrive/vit_gpu_model_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved to: {save_path}")


🚀 Using device: cuda
⏱ Start training: 2025-11-10 10:01:27.378970
[Epoch 1/1] Train Loss: 2.9501, Train Acc: 0.1719 | Val Loss: 2.2921, Val Acc: 0.1250
✅ Training completed. Total time: 0:00:02.510926
💾 Model saved to: /content/drive/MyDrive/vit_gpu_model_20251110_1001.pth


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
save_path = f"/content/drive/MyDrive/vit_gpu_model_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved to: {save_path}")


💾 Model saved to: /content/drive/MyDrive/vit_gpu_model_20251110_1001.pth


In [7]:
# =====================================================
# ✅ Google Drive 마운트 + 모델 저장 (GPU 학습 코드 포함)
# =====================================================

# 1️⃣ Google Drive 연결
from google.colab import drive
drive.mount('/content/drive')

# 2️⃣ 라이브러리 임포트
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import timm
from torchvision import transforms, datasets

# =====================================================
# ✅ 경로 및 하이퍼파라미터 설정
# =====================================================
DATA_PATH = "/content/drive/MyDrive/"
TRAIN_PATH = os.path.join(DATA_PATH, "train_images")
TEST_PATH = os.path.join(DATA_PATH, "test.jpg")
MODEL_PATH = os.path.join(DATA_PATH, "jx_vit_base_p16_224-80ecf9dd.pth")

IMG_SIZE = 224
BATCH_SIZE = 16
LR = 2e-5
N_EPOCHS = 1
GAMMA = 0.7

# =====================================================
# ✅ 데이터 변환 정의
# =====================================================
transforms_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transforms_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ✅ 예시용 dummy 데이터 (사용자 데이터셋으로 대체 가능)
train_dataset = datasets.FakeData(
    size=64, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_train
)
valid_dataset = datasets.FakeData(
    size=16, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_valid
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =====================================================
# ✅ ViT 모델 정의
# =====================================================
class ViTBase16(nn.Module):
    def __init__(self, n_classes, pretrained=False):
        super(ViTBase16, self).__init__()
        self.model = timm.create_model("vit_base_patch16_224", pretrained=False)
        if pretrained:
            self.model.load_state_dict(torch.load(MODEL_PATH))
        in_features = self.model.head.in_features
        self.model.head = nn.Linear(in_features, n_classes)

    def forward(self, x):
        return self.model(x)

model = ViTBase16(n_classes=5, pretrained=False)

# =====================================================
# ✅ GPU 설정 및 학습 준비
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=GAMMA)

# =====================================================
# ✅ 학습 및 검증 함수
# =====================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        acc = (outputs.argmax(1) == labels).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            acc = (outputs.argmax(1) == labels).float().mean()
            total_loss += loss.item()
            total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)

# =====================================================
# ✅ 학습 루프
# =====================================================
start_time = datetime.now()
print(f"⏱ Start training: {start_time}")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, valid_loader, criterion, device)
    scheduler.step()

    print(f"[Epoch {epoch+1}/{N_EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"✅ Training completed. Total time: {datetime.now() - start_time}")

# =====================================================
# ✅ 모델 저장
# =====================================================
save_path = f"/content/drive/MyDrive/vit_gpu_model_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved to: {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Using device: cuda
⏱ Start training: 2025-11-10 10:02:33.541356
[Epoch 1/1] Train Loss: 2.5343, Train Acc: 0.2656 | Val Loss: 2.3074, Val Acc: 0.3125
✅ Training completed. Total time: 0:00:02.494859
💾 Model saved to: /content/drive/MyDrive/vit_gpu_model_20251110_1002.pth


In [8]:
# =====================================================
# ✅ 0. Google Drive 마운트
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

# =====================================================
# ✅ 1. 라이브러리 임포트
# =====================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datetime import datetime
import timm
from torchvision import transforms, datasets

# =====================================================
# ✅ 2. 경로 및 하이퍼파라미터 설정
# =====================================================
DATA_PATH = "/content/drive/MyDrive/"
TRAIN_PATH = os.path.join(DATA_PATH, "train_images")
TEST_PATH = os.path.join(DATA_PATH, "test.jpg")
MODEL_PATH = os.path.join(DATA_PATH, "jx_vit_base_p16_224-80ecf9dd.pth")

IMG_SIZE = 224
BATCH_SIZE = 16
LR = 2e-5
N_EPOCHS = 1
GAMMA = 0.7

# =====================================================
# ✅ 3. 데이터 변환 정의
# =====================================================
transforms_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transforms_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# 예시용 더미 데이터셋 (사용자는 실제 데이터로 대체 가능)
train_dataset = datasets.FakeData(
    size=64, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_train
)
valid_dataset = datasets.FakeData(
    size=16, image_size=(3, IMG_SIZE, IMG_SIZE), num_classes=5, transform=transforms_valid
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =====================================================
# ✅ 4. Vision Transformer 모델 정의
# =====================================================
class ViTBase16(nn.Module):
    def __init__(self, n_classes, pretrained=False):
        super(ViTBase16, self).__init__()
        # timm에서 ViT Base 모델 불러오기
        self.model = timm.create_model("vit_base_patch16_224", pretrained=False)
        if pretrained:
            self.model.load_state_dict(torch.load(MODEL_PATH))
        in_features = self.model.head.in_features
        # ImageNet(1000-class) head → 사용자 데이터셋용 head로 교체
        self.model.head = nn.Linear(in_features, n_classes)

    def forward(self, x):
        return self.model(x)

model = ViTBase16(n_classes=5, pretrained=False)

# =====================================================
# ✅ 5. GPU 설정 및 학습 준비
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=GAMMA)

# =====================================================
# ✅ 6. 학습 함수
# =====================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        acc = (outputs.argmax(1) == labels).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)

# =====================================================
# ✅ 7. 검증 함수
# =====================================================
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            acc = (outputs.argmax(1) == labels).float().mean()
            total_loss += loss.item()
            total_acc += acc.item()
    return total_loss / len(loader), total_acc / len(loader)

# =====================================================
# ✅ 8. 학습 루프
# =====================================================
start_time = datetime.now()
print(f"⏱ Start training: {start_time}")

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, valid_loader, criterion, device)
    scheduler.step()

    print(f"[Epoch {epoch+1}/{N_EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"✅ Training completed. Total time: {datetime.now() - start_time}")

# =====================================================
# ✅ 9. 모델 저장
# =====================================================
save_path = f"/content/drive/MyDrive/vit_gpu_model_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved to: {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Using device: cuda
⏱ Start training: 2025-11-10 10:03:58.876989
[Epoch 1/1] Train Loss: 2.7988, Train Acc: 0.2344 | Val Loss: 1.9848, Val Acc: 0.1250
✅ Training completed. Total time: 0:00:02.527591
💾 Model saved to: /content/drive/MyDrive/vit_gpu_model_20251110_1004.pth


In [9]:
# =====================================================
# ✅ 0. Google Drive 마운트
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

# =====================================================
# ✅ 1. 라이브러리 임포트
# =====================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datetime import datetime
import os

# =====================================================
# ✅ 2. 경로 및 하이퍼파라미터 설정
# =====================================================
DATA_PATH = "/content/drive/MyDrive/"
IMG_SIZE = 64
BATCH_SIZE = 32
LR = 1e-3
EPOCHS = 3
NUM_CLASSES = 10  # 예: CIFAR-10 데이터 기준

# =====================================================
# ✅ 3. 데이터셋 로드 및 전처리
# =====================================================
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transform_valid = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ⚠️ 예시로 CIFAR-10 사용 (사용자 데이터로 대체 가능)
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
valid_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_valid)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# =====================================================
# ✅ 4. 합성곱 신경망(CNN) 모델 정의
# =====================================================
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE // 8) * (IMG_SIZE // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

# 모델 생성
model = SimpleCNN(num_classes=NUM_CLASSES)

# =====================================================
# ✅ 5. GPU 설정 및 손실함수/옵티마이저 정의
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# =====================================================
# ✅ 6. 학습 함수 정의
# =====================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc = 0.0, 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        acc = (outputs.argmax(1) == labels).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

# =====================================================
# ✅ 7. 검증 함수 정의
# =====================================================
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0.0, 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            acc = (outputs.argmax(1) == labels).float().mean()
            total_loss += loss.item()
            total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

# =====================================================
# ✅ 8. 학습 루프
# =====================================================
start_time = datetime.now()
print(f"⏱ Start training: {start_time}")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, valid_loader, criterion, device)
    print(f"[Epoch {epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"✅ Training completed. Total time: {datetime.now() - start_time}")

# =====================================================
# ✅ 9. 모델 저장
# =====================================================
save_path = f"/content/drive/MyDrive/simple_cnn_model_{datetime.now().strftime('%Y%m%d_%H%M')}.pth"
torch.save(model.state_dict(), save_path)
print(f"💾 Model saved to: {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 170M/170M [00:31<00:00, 5.37MB/s]


🚀 Using device: cuda
⏱ Start training: 2025-11-10 10:05:02.330577
[Epoch 1/3] Train Loss: 1.5054, Train Acc: 0.4521 | Val Loss: 1.2003, Val Acc: 0.5671
[Epoch 2/3] Train Loss: 1.1440, Train Acc: 0.5911 | Val Loss: 0.9918, Val Acc: 0.6480
[Epoch 3/3] Train Loss: 0.9863, Train Acc: 0.6527 | Val Loss: 0.9011, Val Acc: 0.6861
✅ Training completed. Total time: 0:01:11.579418
💾 Model saved to: /content/drive/MyDrive/simple_cnn_model_20251110_1006.pth
